# Chapter 7: Socio-Technical Transformation of Algo Trading

Companion notebook for Chapter 7 of *Mastering AI-Powered Finance* (`Overleaf/Book 1`). It was built from Chapter 6 of the full manuscript (AI and Algorithmic Trading Strategies); the source references below point to that manuscript.

Notebook extracted from `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`.

This notebook turns the chapter code examples into a student-facing workflow. Code cells follow the order used in the chapter; Markdown cells identify the source section and preserve sample output blocks from the LaTeX when present.

Run the notebook from top to bottom. Some examples are intentionally small and synthetic; examples that call external services require the dependency or credential noted below.

**External requirements for some cells:** internet market-data access.

**Review notes.** The companion retains the chapter's full progression, with corrected execution timing: signals use completed daily bars, trade at the next session's open, and earn the return until the following open. Training includes only labels already realized at each refit. Adjusted historical OHLC data approximates dividend-inclusive returns; it is not a point-in-time vendor archive or an executable fill history. Cash earns zero and the illustrative cost is 10 bp **per side** (20 bp per entry/exit round trip). Opening-auction slippage, taxes, and market impact require separate modeling. Historical output blocks below are retained as manuscript illustrations, not expected results of the corrected notebook.
Manuscript synchronization (2026-09-21): concise printed algorithms now match this expanded notebook. Corrected live-data output replaces obsolete tables; these review-run observations are not fixed targets for future downloads.


In [1]:
# Common notebook setup: keep convergence and data warnings visible.
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
# Grayscale defaults, matching the printed book: black, mid and light gray,
# each with its own line style so series stay distinguishable in print.
from cycler import cycler
plt.rcParams['axes.prop_cycle'] = cycler(color=['0.0', '0.45', '0.7'], linestyle=['-', '--', ':'])
plt.rcParams['image.cmap'] = 'Greys'

## AI Tutorial: Building an ML-Enhanced Trading System / Phase 1: Feature Engineering

### Code example 1
Source line: `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load the bundled price snapshot. Yahoo serves back-adjusted history at
# float32 precision and its adjustment factor shifts slightly between
# requests. Those differences are economically meaningless but they move
# tree split points, so the model results below would not reproduce from a
# fresh download. Set REFRESH_MARKET_DATA=1 to pull current data instead.
SNAPSHOT = Path('data/spy_2015_2024.csv')
if SNAPSHOT.exists() and os.environ.get('REFRESH_MARKET_DATA') != '1':
    data = pd.read_csv(SNAPSHOT, index_col=0, parse_dates=True)
else:
    data = yf.download('SPY', start='2015-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)
    if data is None or data.empty:
        raise RuntimeError('SPY download returned no data; retry the market-data request.')
    if isinstance(data.columns, pd.MultiIndex):
        ticker_level = next(i for i in range(data.columns.nlevels)
                            if 'SPY' in data.columns.get_level_values(i))
        data = data.xs('SPY', level=ticker_level, axis=1)
required = ['Open', 'High', 'Low', 'Close', 'Volume']
if data.empty or not set(required).issubset(data.columns):
    raise RuntimeError('SPY download is empty or incomplete; retry the market-data request.')
data = data.sort_index()
if data.index.has_duplicates or not np.isfinite(data[required].to_numpy()).all():
    raise ValueError('Expected unique sessions and finite OHLCV; inspect vendor data before use.')
if (data[['Open', 'High', 'Low', 'Close']] <= 0).any().any() or (data['Volume'] < 0).any():
    raise ValueError('Expected positive prices and nonnegative volume.')
print(f'Loaded {len(data)} trading days for SPY')

def engineer_features(df):
    features = pd.DataFrame(index=df.index)
    close, high, low, volume = df['Close'], df['High'], df['Low'], df['Volume']

    # Trend: price position relative to moving averages
    for w in [5, 10, 20, 50]:
        features[f'close_to_sma_{w}'] = close / close.rolling(w).mean() - 1

    # MACD
    ema12 = close.ewm(span=12, adjust=False).mean()
    ema26 = close.ewm(span=26, adjust=False).mean()
    features['macd'] = ema12 - ema26
    features['macd_signal'] = features['macd'].ewm(span=9, adjust=False).mean()
    features['macd_hist'] = features['macd'] - features['macd_signal']

    # RSI (14-day)
    delta = close.diff()
    gain = delta.where(delta > 0, 0.0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0.0)).rolling(14).mean()
    features['rsi_14'] = (100 - (100 / (1 + gain / loss)))
    features.loc[(gain == 0) & (loss == 0), 'rsi_14'] = 50.0

    # Bollinger Band position
    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    features['bb_position'] = ((close - bb_mid) / (2 * bb_std)).where(bb_std != 0, 0.0)

    # Volatility
    tr = pd.concat([high-low, (high-close.shift(1)).abs(),
                    (low-close.shift(1)).abs()], axis=1).max(axis=1)
    features['atr_ratio'] = tr.rolling(14).mean() / close
    features['volatility_20d'] = close.pct_change(fill_method=None).rolling(20).std()*np.sqrt(252)

    # Volume and lagged returns
    features['volume_sma_ratio'] = volume / volume.rolling(20).mean()
    for lag in [1, 2, 3, 5, 10, 20]:
        features[f'return_{lag}d'] = close.pct_change(lag, fill_method=None)

    return features.replace([np.inf, -np.inf], np.nan)

features = engineer_features(data)
print(f"Engineered {len(features.columns)} features")

Loaded 2515 trading days for SPY
Engineered 18 features


### Corrected review-run output

Observed run on the bundled price snapshot, used for the synchronized manuscript figures. Pinned data makes this reproducible; library differences can still change subsequent results.

```text
Downloaded 2515 trading days for SPY
Engineered 18 features
```


## AI Tutorial: Building an ML-Enhanced Trading System / Phase 2: Walk-Forward Validation and Model Training

### Code example 2
Source line: `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`

In [3]:
# Signal dated t is formed after close t; trade open t+1 to open t+2.
next_return = data['Open'].shift(-2) / data['Open'].shift(-1) - 1
label_end = pd.Series(data.index, index=data.index).shift(-2)
target = (next_return > 0).astype(float).where(next_return.notna())
dataset = features.join(target.rename('target')).join(
    next_return.rename('next_return')).join(label_end.rename('label_end')).dropna()
feature_cols = list(features.columns)

def walk_forward(dataset, feature_cols, model_class, model_params,
                 train_size=504, step=21):
    if train_size < 1 or step < 1:
        raise ValueError('train_size and step must be positive.')
    if not dataset.index.is_monotonic_increasing or dataset.index.has_duplicates:
        raise ValueError('Dataset sessions must be sorted and unique.')
    # One extra signal row is needed: the immediately preceding row's
    # open-to-open label is still unknown at today's close.
    first_test = train_size + 1
    if len(dataset) <= first_test:
        raise ValueError('Not enough observations for training and a test period.')
    predictions, actuals, dates = [], [], []
    for start in range(first_test, len(dataset), step):
        candidates = dataset.iloc[:start]
        train = candidates.loc[candidates['label_end'] <= dataset.index[start]]
        test = dataset.iloc[start:start+step]  # includes the final partial block
        model = model_class(**model_params)
        if isinstance(model, LogisticRegression):
            model = make_pipeline(StandardScaler(), model)
        model.fit(train[feature_cols], train['target'])
        preds = model.predict(test[feature_cols])
        predictions.extend(preds); actuals.extend(test['target'])
        dates.extend(test.index)
    return pd.DataFrame({'prediction': predictions, 'actual': actuals},
                        index=pd.DatetimeIndex(dates, name=dataset.index.name))

# Compare three model families. Scaling is fitted anew on each training fold.
models = {
    'Logistic Regression': (LogisticRegression,
        {'max_iter': 1000, 'C': 0.1}),
    'Random Forest': (RandomForestClassifier,
        {'n_estimators': 100, 'max_depth': 5, 'min_samples_leaf': 50,
         'random_state': 42}),
    'XGBoost': (XGBClassifier,
        {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1,
         'verbosity': 0, 'random_state': 42, 'n_jobs': 1}),
}
results = {}
for name, (cls, params) in models.items():
    res = walk_forward(dataset, feature_cols, cls, params)
    results[name] = res
    baseline = accuracy_score(res['actual'], np.ones(len(res)))
    print(f"{name}: accuracy = {accuracy_score(res['actual'], res['prediction']):.4f}; "
          f"always-up baseline on same dates = {baseline:.4f}; n = {len(res)}")

Logistic Regression: accuracy = 0.5533; always-up baseline on same dates = 0.5666; n = 1959


Random Forest: accuracy = 0.5518; always-up baseline on same dates = 0.5666; n = 1959


XGBoost: accuracy = 0.5135; always-up baseline on same dates = 0.5666; n = 1959


### Corrected review-run output

Observed run on the bundled price snapshot, used for the synchronized manuscript figures. Pinned data makes this reproducible; library differences can still change subsequent results.

```text
Logistic Regression: accuracy = 0.5533; always-up baseline on same dates = 0.5666; n = 1959
Random Forest: accuracy = 0.5518; always-up baseline on same dates = 0.5666; n = 1959
XGBoost: accuracy = 0.5135; always-up baseline on same dates = 0.5666; n = 1959
```


## AI Tutorial: Building an ML-Enhanced Trading System / Phase 3: Backtesting with Transaction Costs

### Code example 3
Source line: `Overleaf/Final Project/Chapter 6 - AI and Algorithmic Trading Strategies/main.tex`

In [4]:
def backtest(preds_df, dataset, tc=0.001):
    """Long/cash open-to-open returns; tc is a proportional cost per side."""
    if not 0 <= tc < 1:
        raise ValueError('tc must lie in [0, 1).')
    if preds_df.empty or not preds_df.index.is_unique or not preds_df.index.is_monotonic_increasing:
        raise ValueError('Predictions must have sorted, unique, nonempty dates.')
    locations = dataset.index.get_indexer(preds_df.index)
    if (locations < 0).any() or (np.diff(locations) != 1).any():
        raise ValueError('Predictions must cover consecutive dataset sessions.')
    aligned = preds_df.join(dataset[['next_return']], how='left')
    if not aligned['prediction'].isin([0, 1]).all() or not np.isfinite(aligned['next_return']).all():
        raise ValueError('Expected binary predictions and finite forward returns.')
    position = aligned['prediction'].astype(float)
    # Start in cash, so the initial long entry incurs a fee as well.
    turnover = position.diff().abs()
    turnover.iloc[0] = position.iloc[0]
    # Fees are applied before each holding-period return; liquidate at the
    # final open and charge that closing fee on the remaining equity.
    net = (1 - turnover * tc) * (1 + position * aligned['next_return']) - 1
    net.iloc[-1] = (1 + net.iloc[-1]) * (1 - position.iloc[-1] * tc) - 1
    return net.rename('net')

def performance_metrics(net):
    if net.empty or not np.isfinite(net).all() or (net <= -1).any():
        raise ValueError('Expected finite nonempty returns above -100%.')
    cum = (1 + net).cumprod()
    vol = net.std() * np.sqrt(252)
    # Include starting equity 1.0 when measuring the first loss/drawdown.
    peaks = cum.cummax().clip(lower=1.0)
    return {'CAGR': cum.iloc[-1] ** (252 / len(net)) - 1,
            'Volatility': vol,
            'Sharpe (rf=0)': net.mean() * 252 / vol if vol > 0 else np.nan,
            'Max Drawdown': (cum / peaks - 1).min()}

# Buy & hold uses identical dates and entry/exit cost assumptions.
comparison = {}
for name, res in results.items():
    comparison[name] = performance_metrics(backtest(res, dataset))
benchmark = pd.DataFrame({'prediction': 1}, index=next(iter(results.values())).index)
comparison['Buy & Hold'] = performance_metrics(backtest(benchmark, dataset))
performance = pd.DataFrame(comparison)
print(performance.round(4))

               Logistic Regression  Random Forest  XGBoost  Buy & Hold
CAGR                        0.1052         0.0973   0.0130      0.1419
Volatility                  0.1638         0.1730   0.1494      0.1757
Sharpe (rf=0)               0.6929         0.6240   0.1614      0.8439
Max Drawdown               -0.2702        -0.3205  -0.3327     -0.3205


### Corrected review-run output

Observed run on the bundled price snapshot, used for the synchronized manuscript figures. Pinned data makes this reproducible; library differences can still change subsequent results.

```text
               Logistic Regression  Random Forest  XGBoost  Buy & Hold
CAGR                        0.1052         0.0973   0.0130      0.1419
Volatility                  0.1638         0.1730   0.1494      0.1757
Sharpe (rf=0)               0.6929         0.6240   0.1614      0.8439
Max Drawdown               -0.2702        -0.3205  -0.3327     -0.3205
```


### Feature-importance interpretation

The manuscript figure uses the final walk-forward Random Forest training fold. Impurity importance is a fitted-model diagnostic, not held-out evidence of predictive contribution. The regeneration script is `reviews/synchronization/regenerate_chapter06_figures.py`.


## Where the tutorial continues

The generative-AI part of this tutorial, an LLM news-sentiment signal that can be added to the features above, is in the Chapter 6 notebook.